# 04 Validate Semantic Model

This notebook adds a **hybrid validation layer** to the project.

It combines:

- **Great Expectations (GX)** for expectation-style validation
  - uniqueness
  - null checks
  - row-count checks
- **Pandas** for cross-table validation
  - referential integrity
  - row-count reconciliation

## Why use a hybrid approach?

Great Expectations is well suited to defining explicit expectations about a single dataset, including null checks, uniqueness, and row-count checks. GX Core also supports validating data in Pandas DataFrames by creating a Data Context, a pandas Data Source, a Data Asset, and a Batch that receives the DataFrame at runtime. Great Expectations also documents filesystem-based data access for local files and cloud storage.  
The cross-table integrity checks in this notebook are kept in pandas because they are easier to express clearly at this stage of the project. 

## Input files

This notebook reads processed CSV files from `data/processed/`:

- `dim_date.csv`
- `dim_user.csv`
- `dim_report.csv`
- `dim_page.csv`
- `fact_report_views.csv`
- `fact_page_views.csv`
- `fact_report_loads.csv`

## Output files

This notebook writes validation outputs to `outputs/validation/`:

- `gx_validation_results.csv`
- `referential_integrity_checks.csv`
- `row_count_reconciliation.csv`
- `validation_summary.csv`




## Setup

The notebook is expected to live inside the `notebooks/` folder, so we move one level up to find the project root and then define the processed and validation output folders.


In [1]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed"
VALIDATION_PATH = PROJECT_ROOT / "outputs" / "validation"

VALIDATION_PATH.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Processed path:", PROCESSED_PATH)
print("Validation path:", VALIDATION_PATH)

Project root: /Users/masegomodibane/Documents/GitHub/Data Science Projects /Forecasting Report Usage/GitHub Final Version/report-usage-forecasting
Processed path: /Users/masegomodibane/Documents/GitHub/Data Science Projects /Forecasting Report Usage/GitHub Final Version/report-usage-forecasting/data/processed
Validation path: /Users/masegomodibane/Documents/GitHub/Data Science Projects /Forecasting Report Usage/GitHub Final Version/report-usage-forecasting/outputs/validation


## Load processed tables


In [2]:
dim_date = pd.read_csv(PROCESSED_PATH / "dim_date.csv", parse_dates=["date", "week_start_date"])
dim_user = pd.read_csv(PROCESSED_PATH / "dim_user.csv")
dim_report = pd.read_csv(PROCESSED_PATH / "dim_report.csv")
dim_page = pd.read_csv(PROCESSED_PATH / "dim_page.csv")

fact_report_views = pd.read_csv(PROCESSED_PATH / "fact_report_views.csv")
fact_page_views = pd.read_csv(PROCESSED_PATH / "fact_page_views.csv")
fact_report_loads = pd.read_csv(PROCESSED_PATH / "fact_report_loads.csv")

print("Loaded processed tables successfully.")

Loaded processed tables successfully.


In [3]:
print("dim_date:", dim_date.shape)
print("dim_user:", dim_user.shape)
print("dim_report:", dim_report.shape)
print("dim_page:", dim_page.shape)
print("fact_report_views:", fact_report_views.shape)
print("fact_page_views:", fact_page_views.shape)
print("fact_report_loads:", fact_report_loads.shape)

dim_date: (455, 6)
dim_user: (200, 3)
dim_report: (30, 5)
dim_page: (150, 4)
fact_report_views: (411006, 6)
fact_page_views: (879149, 7)
fact_report_loads: (411006, 7)


## Initialize Great Expectations

GX Core validates data by creating a Data Context, then connecting data through a Data Source, Data Asset, Batch Definition, and Batch.  
For this notebook, we use a **Pandas DataFrame workflow** so each processed table can be validated directly from memory. GX documents this exact DataFrame pattern with `context.data_sources.add_pandas(...)`, `add_dataframe_asset(...)`, and `add_batch_definition_whole_dataframe(...)`.


In [4]:
import great_expectations as gx

print("GX version:", gx.__version__)

context = gx.get_context()
print("GX context initialized.")

GX version: 1.16.1
GX context initialized.


## Helper functions

We use three helpers:

- `get_batch(...)` to create or retrieve a GX Batch for a DataFrame
- `run_gx_expectations(...)` to execute GX expectations and collect tabular results
- `parse_gx_result(...)` to flatten validation output into a DataFrame


In [5]:
def get_or_create_pandas_datasource(context, datasource_name="pandas_validation_source"):
    try:
        return context.data_sources.get(datasource_name)
    except Exception:
        return context.data_sources.add_pandas(datasource_name)


def get_or_create_dataframe_asset(data_source, asset_name):
    try:
        return data_source.get_asset(asset_name)
    except Exception:
        return data_source.add_dataframe_asset(name=asset_name)


def get_or_create_batch_definition(data_asset, batch_definition_name="whole_dataframe"):
    try:
        return data_asset.get_batch_definition(batch_definition_name)
    except Exception:
        return data_asset.add_batch_definition_whole_dataframe(batch_definition_name)


def get_batch(context, df, asset_name):
    data_source = get_or_create_pandas_datasource(context)
    data_asset = get_or_create_dataframe_asset(data_source, asset_name)
    batch_definition = get_or_create_batch_definition(data_asset)
    batch = batch_definition.get_batch(batch_parameters={"dataframe": df})
    return batch

In [6]:
def parse_gx_result(table_name, check_group, expectation_name, result_obj):
    success = result_obj.get("success", None)
    expectation_config = result_obj.get("expectation_config", {})
    result = result_obj.get("result", {})

    return {
        "table_name": table_name,
        "check_group": check_group,
        "expectation_name": expectation_name,
        "success": success,
        "unexpected_count": result.get("unexpected_count"),
        "unexpected_percent": result.get("unexpected_percent"),
        "missing_count": result.get("missing_count"),
        "missing_percent": result.get("missing_percent"),
        "element_count": result.get("element_count"),
        "details": str(expectation_config.get("kwargs", {}))
    }


def run_gx_expectations(context, df, table_name, expectations, check_group):
    batch = get_batch(context, df, asset_name=f"{table_name}_asset")
    records = []

    for expectation in expectations:
        result = batch.validate(expectation)
        expectation_name = expectation.__class__.__name__
        records.append(
            parse_gx_result(
                table_name=table_name,
                check_group=check_group,
                expectation_name=expectation_name,
                result_obj=result
            )
        )

    return pd.DataFrame(records)

## GX uniqueness checks


In [7]:
gx_uniqueness_results = []

uniqueness_specs = {
    "dim_date": (dim_date, [gx.expectations.ExpectColumnValuesToBeUnique(column="date_key")]),
    "dim_user": (dim_user, [gx.expectations.ExpectColumnValuesToBeUnique(column="user_key")]),
    "dim_report": (dim_report, [gx.expectations.ExpectColumnValuesToBeUnique(column="report_id")]),
    "dim_page": (dim_page, [gx.expectations.ExpectColumnValuesToBeUnique(column="page_key")]),
}

for table_name, (df, expectations) in uniqueness_specs.items():
    gx_uniqueness_results.append(
        run_gx_expectations(context, df, table_name, expectations, check_group="uniqueness")
    )

gx_uniqueness_results = pd.concat(gx_uniqueness_results, ignore_index=True)
gx_uniqueness_results

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

,table_name,check_group,expectation_name,success,unexpected_count,unexpected_percent,missing_count,missing_percent,element_count,details
0,dim_date,uniqueness,ExpectColumnValuesToBeUnique,True,0,0.0,0,0.0,455,{'batch_id': 'pandas_validation_source-dim_dat...
1,dim_user,uniqueness,ExpectColumnValuesToBeUnique,True,0,0.0,0,0.0,200,{'batch_id': 'pandas_validation_source-dim_use...
2,dim_report,uniqueness,ExpectColumnValuesToBeUnique,True,0,0.0,0,0.0,30,{'batch_id': 'pandas_validation_source-dim_rep...
3,dim_page,uniqueness,ExpectColumnValuesToBeUnique,True,0,0.0,0,0.0,150,{'batch_id': 'pandas_validation_source-dim_pag...


## GX null checks


In [8]:
gx_null_results = []

null_specs = {
    "dim_date": (dim_date, [
        gx.expectations.ExpectColumnValuesToNotBeNull(column="date_key")
    ]),
    "dim_user": (dim_user, [
        gx.expectations.ExpectColumnValuesToNotBeNull(column="user_key")
    ]),
    "dim_report": (dim_report, [
        gx.expectations.ExpectColumnValuesToNotBeNull(column="report_id")
    ]),
    "dim_page": (dim_page, [
        gx.expectations.ExpectColumnValuesToNotBeNull(column="page_key"),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="report_id"),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="section_id")
    ]),
    "fact_report_views": (fact_report_views, [
        gx.expectations.ExpectColumnValuesToNotBeNull(column="date_key"),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="report_id"),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="user_key"),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="view_count")
    ]),
    "fact_page_views": (fact_page_views, [
        gx.expectations.ExpectColumnValuesToNotBeNull(column="date_key"),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="report_id"),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="page_key"),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="user_key"),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="page_view_count")
    ]),
    "fact_report_loads": (fact_report_loads, [
        gx.expectations.ExpectColumnValuesToNotBeNull(column="date_key"),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="report_id"),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="user_key"),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="load_time_ms")
    ]),
}

for table_name, (df, expectations) in null_specs.items():
    gx_null_results.append(
        run_gx_expectations(context, df, table_name, expectations, check_group="nulls")
    )

gx_null_results = pd.concat(gx_null_results, ignore_index=True)
gx_null_results.head()

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

...

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

,table_name,check_group,expectation_name,success,unexpected_count,unexpected_percent,missing_count,missing_percent,element_count,details
0,dim_date,nulls,ExpectColumnValuesToNotBeNull,True,0,0.0,None,None,455,{'batch_id': 'pandas_validation_source-dim_dat...
1,dim_user,nulls,ExpectColumnValuesToNotBeNull,True,0,0.0,None,None,200,{'batch_id': 'pandas_validation_source-dim_use...
2,dim_report,nulls,ExpectColumnValuesToNotBeNull,True,0,0.0,None,None,30,{'batch_id': 'pandas_validation_source-dim_rep...
3,dim_page,nulls,ExpectColumnValuesToNotBeNull,True,0,0.0,None,None,150,{'batch_id': 'pandas_validation_source-dim_pag...
4,dim_page,nulls,ExpectColumnValuesToNotBeNull,True,0,0.0,None,None,150,{'batch_id': 'pandas_validation_source-dim_pag...


## GX row-count checks

GX documents row-count and data-volume expectations as a core validation use case.  
For the first pass, we keep the row-count rule simple: each table must have at least one row. 


In [9]:
gx_rowcount_results = []

rowcount_specs = {
    "dim_date": dim_date,
    "dim_user": dim_user,
    "dim_report": dim_report,
    "dim_page": dim_page,
    "fact_report_views": fact_report_views,
    "fact_page_views": fact_page_views,
    "fact_report_loads": fact_report_loads,
}

for table_name, df in rowcount_specs.items():
    expectations = [
        gx.expectations.ExpectTableRowCountToBeBetween(min_value=1)
    ]
    gx_rowcount_results.append(
        run_gx_expectations(context, df, table_name, expectations, check_group="row_count")
    )

gx_rowcount_results = pd.concat(gx_rowcount_results, ignore_index=True)
gx_rowcount_results

Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

,table_name,check_group,expectation_name,success,unexpected_count,unexpected_percent,missing_count,missing_percent,element_count,details
0,dim_date,row_count,ExpectTableRowCountToBeBetween,True,None,None,None,None,None,{'batch_id': 'pandas_validation_source-dim_dat...
1,dim_user,row_count,ExpectTableRowCountToBeBetween,True,None,None,None,None,None,{'batch_id': 'pandas_validation_source-dim_use...
2,dim_report,row_count,ExpectTableRowCountToBeBetween,True,None,None,None,None,None,{'batch_id': 'pandas_validation_source-dim_rep...
3,dim_page,row_count,ExpectTableRowCountToBeBetween,True,None,None,None,None,None,{'batch_id': 'pandas_validation_source-dim_pag...
4,fact_report_views,row_count,ExpectTableRowCountToBeBetween,True,None,None,None,None,None,{'batch_id': 'pandas_validation_source-fact_re...
5,fact_page_views,row_count,ExpectTableRowCountToBeBetween,True,None,None,None,None,None,{'batch_id': 'pandas_validation_source-fact_pa...
6,fact_report_loads,row_count,ExpectTableRowCountToBeBetween,True,None,None,None,None,None,{'batch_id': 'pandas_validation_source-fact_re...


## Combine GX results


In [10]:
gx_validation_results = pd.concat(
    [gx_uniqueness_results, gx_null_results, gx_rowcount_results],
    ignore_index=True
)

gx_validation_results.head(20)

,table_name,check_group,expectation_name,success,unexpected_count,unexpected_percent,missing_count,missing_percent,element_count,details
0,dim_date,uniqueness,ExpectColumnValuesToBeUnique,True,0,0.0,0,0.0,455,{'batch_id': 'pandas_validation_source-dim_dat...
1,dim_user,uniqueness,ExpectColumnValuesToBeUnique,True,0,0.0,0,0.0,200,{'batch_id': 'pandas_validation_source-dim_use...
2,dim_report,uniqueness,ExpectColumnValuesToBeUnique,True,0,0.0,0,0.0,30,{'batch_id': 'pandas_validation_source-dim_rep...
3,dim_page,uniqueness,ExpectColumnValuesToBeUnique,True,0,0.0,0,0.0,150,{'batch_id': 'pandas_validation_source-dim_pag...
4,dim_date,nulls,ExpectColumnValuesToNotBeNull,True,0,0.0,None,None,455,{'batch_id': 'pandas_validation_source-dim_dat...
5,dim_user,nulls,ExpectColumnValuesToNotBeNull,True,0,0.0,None,None,200,{'batch_id': 'pandas_validation_source-dim_use...
6,dim_report,nulls,ExpectColumnValuesToNotBeNull,True,0,0.0,None,None,30,{'batch_id': 'pandas_validation_source-dim_rep...
7,dim_page,nulls,ExpectColumnValuesToNotBeNull,True,0,0.0,None,None,150,{'batch_id': 'pandas_validation_source-dim_pag...
8,dim_page,nulls,ExpectColumnValuesToNotBeNull,True,0,0.0,None,None,150,{'batch_id': 'pandas_validation_source-dim_pag...
9,dim_page,nulls,ExpectColumnValuesToNotBeNull,True,0,0.0,None,None,150,{'batch_id': 'pandas_validation_source-dim_pag...


## Referential integrity checks with pandas

This section checks whether fact-table foreign keys exist in the corresponding dimension tables.


In [11]:
ri_checks = []

def add_ri_check(check_name, fact_df, fact_key, dim_df, dim_key):
    invalid_mask = ~fact_df[fact_key].isin(dim_df[dim_key])
    invalid_count = int(invalid_mask.sum())
    total_rows = int(len(fact_df))

    ri_checks.append({
        "check_name": check_name,
        "fact_key": fact_key,
        "dimension_key": dim_key,
        "invalid_count": invalid_count,
        "total_rows": total_rows,
        "success": invalid_count == 0
    })

add_ri_check("fact_report_views.report_id -> dim_report.report_id",
             fact_report_views, "report_id", dim_report, "report_id")
add_ri_check("fact_report_views.user_key -> dim_user.user_key",
             fact_report_views, "user_key", dim_user, "user_key")
add_ri_check("fact_report_views.date_key -> dim_date.date_key",
             fact_report_views, "date_key", dim_date, "date_key")

add_ri_check("fact_page_views.report_id -> dim_report.report_id",
             fact_page_views, "report_id", dim_report, "report_id")
add_ri_check("fact_page_views.page_key -> dim_page.page_key",
             fact_page_views, "page_key", dim_page, "page_key")
add_ri_check("fact_page_views.user_key -> dim_user.user_key",
             fact_page_views, "user_key", dim_user, "user_key")
add_ri_check("fact_page_views.date_key -> dim_date.date_key",
             fact_page_views, "date_key", dim_date, "date_key")

add_ri_check("fact_report_loads.report_id -> dim_report.report_id",
             fact_report_loads, "report_id", dim_report, "report_id")
add_ri_check("fact_report_loads.user_key -> dim_user.user_key",
             fact_report_loads, "user_key", dim_user, "user_key")
add_ri_check("fact_report_loads.date_key -> dim_date.date_key",
             fact_report_loads, "date_key", dim_date, "date_key")

referential_integrity_checks = pd.DataFrame(ri_checks)
referential_integrity_checks

,check_name,fact_key,dimension_key,invalid_count,total_rows,success
0,fact_report_views.report_id -> dim_report.repo...,report_id,report_id,0,411006,True
1,fact_report_views.user_key -> dim_user.user_key,user_key,user_key,0,411006,True
2,fact_report_views.date_key -> dim_date.date_key,date_key,date_key,0,411006,True
3,fact_page_views.report_id -> dim_report.report_id,report_id,report_id,0,879149,True
4,fact_page_views.page_key -> dim_page.page_key,page_key,page_key,0,879149,True
5,fact_page_views.user_key -> dim_user.user_key,user_key,user_key,0,879149,True
6,fact_page_views.date_key -> dim_date.date_key,date_key,date_key,0,879149,True
7,fact_report_loads.report_id -> dim_report.repo...,report_id,report_id,0,411006,True
8,fact_report_loads.user_key -> dim_user.user_key,user_key,user_key,0,411006,True
9,fact_report_loads.date_key -> dim_date.date_key,date_key,date_key,0,411006,True


## Row-count reconciliation

This section compares row counts between raw and processed tables.

GX highlights row-count and volume checks as a common data-quality use case. Here, we extend that idea with a simple raw-versus-processed reconciliation table to make transformation behaviour visible.


In [12]:
RAW_PATH = PROJECT_ROOT / "data" / "raw"

raw_reports = pd.read_csv(RAW_PATH / "reports.csv")
raw_users = pd.read_csv(RAW_PATH / "users.csv")
raw_report_pages = pd.read_csv(RAW_PATH / "report_pages.csv")
raw_dates = pd.read_csv(RAW_PATH / "dates.csv")
raw_report_views = pd.read_csv(RAW_PATH / "report_views.csv")
raw_report_page_views = pd.read_csv(RAW_PATH / "report_page_views.csv")
raw_report_load_times = pd.read_csv(RAW_PATH / "report_load_times.csv")

row_count_reconciliation = pd.DataFrame([
    {"raw_table": "reports", "processed_table": "dim_report", "raw_row_count": len(raw_reports), "processed_row_count": len(dim_report)},
    {"raw_table": "users", "processed_table": "dim_user", "raw_row_count": len(raw_users), "processed_row_count": len(dim_user)},
    {"raw_table": "report_pages", "processed_table": "dim_page", "raw_row_count": len(raw_report_pages), "processed_row_count": len(dim_page)},
    {"raw_table": "dates", "processed_table": "dim_date", "raw_row_count": len(raw_dates), "processed_row_count": len(dim_date)},
    {"raw_table": "report_views", "processed_table": "fact_report_views", "raw_row_count": len(raw_report_views), "processed_row_count": len(fact_report_views)},
    {"raw_table": "report_page_views", "processed_table": "fact_page_views", "raw_row_count": len(raw_report_page_views), "processed_row_count": len(fact_page_views)},
    {"raw_table": "report_load_times", "processed_table": "fact_report_loads", "raw_row_count": len(raw_report_load_times), "processed_row_count": len(fact_report_loads)},
])

row_count_reconciliation["row_count_difference"] = (
    row_count_reconciliation["processed_row_count"] - row_count_reconciliation["raw_row_count"]
)

row_count_reconciliation["exact_match"] = (
    row_count_reconciliation["processed_row_count"] == row_count_reconciliation["raw_row_count"]
)

row_count_reconciliation

,raw_table,processed_table,raw_row_count,processed_row_count,row_count_difference,exact_match
0,reports,dim_report,30,30,0,True
1,users,dim_user,200,200,0,True
2,report_pages,dim_page,150,150,0,True
3,dates,dim_date,455,455,0,True
4,report_views,fact_report_views,411006,411006,0,True
5,report_page_views,fact_page_views,879149,879149,0,True
6,report_load_times,fact_report_loads,411006,411006,0,True


## Validation summary


In [13]:
gx_summary = (
    gx_validation_results
    .groupby("check_group", as_index=False)
    .agg(
        checks_run=("success", "count"),
        checks_passed=("success", "sum")
    )
)

gx_summary["checks_failed"] = gx_summary["checks_run"] - gx_summary["checks_passed"]

ri_summary = pd.DataFrame([{
    "check_group": "referential_integrity",
    "checks_run": len(referential_integrity_checks),
    "checks_passed": int(referential_integrity_checks["success"].sum()),
    "checks_failed": int((~referential_integrity_checks["success"]).sum())
}])

reconciliation_summary = pd.DataFrame([{
    "check_group": "row_count_reconciliation",
    "checks_run": len(row_count_reconciliation),
    "checks_passed": int(row_count_reconciliation["exact_match"].sum()),
    "checks_failed": int((~row_count_reconciliation["exact_match"]).sum())
}])

validation_summary = pd.concat(
    [gx_summary, ri_summary, reconciliation_summary],
    ignore_index=True
)

validation_summary

,check_group,checks_run,checks_passed,checks_failed
0,nulls,19,19,0
1,row_count,7,7,0
2,uniqueness,4,4,0
3,referential_integrity,10,10,0
4,row_count_reconciliation,7,7,0


## Save validation outputs


In [14]:
gx_validation_results.to_csv(VALIDATION_PATH / "gx_validation_results.csv", index=False)
referential_integrity_checks.to_csv(VALIDATION_PATH / "referential_integrity_checks.csv", index=False)
row_count_reconciliation.to_csv(VALIDATION_PATH / "row_count_reconciliation.csv", index=False)
validation_summary.to_csv(VALIDATION_PATH / "validation_summary.csv", index=False)

print("Validation outputs saved successfully.")

Validation outputs saved successfully.


## Summary

This notebook gives the project a first reusable validation layer.

It now checks:

- uniqueness
- nulls
- referential integrity
- row-count reconciliation
